# EP3 · DataRetail 360 — Evidencias del pipeline ETL y del almacenamiento analítico

**IDY1102 Estructuras y Visualización de Datos · Sección 003D**
Equipo: Anyara Rosso · Robert Groom · Docente: José Luis Zúñiga Gallardo

Este notebook ejecuta el pipeline de punta a punta y deja la evidencia de cada etapa:

1. perfilado de las fuentes;
2. extracción, transformación y carga por capas **Bronze → Silver → Gold** (arquitectura Medallion);
3. validaciones, manejo de errores, tabla de rechazos y *logging*;
4. modelo dimensional con **SCD tipo 1, 2 y 3**;
5. idempotencia, trazabilidad de una cifra hasta su archivo fuente, optimización y capa semántica para IA.

> El código del pipeline está en `src/`; este notebook solo lo orquesta y muestra los resultados.
> Para reproducirlo: `python -m src.generar_fuentes` y luego *Run All*.

In [ ]:
import json, linecache, shutil, sqlite3, subprocess, sys, time
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import matplotlib.pyplot as plt
import pandas as pd
from sqlalchemy import text

from src import bronze, config, generar_fuentes, gold, pipeline
from src import reglas_calidad as rc

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 180)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

if not (config.FUENTES / "lote_1").exists():
    generar_fuentes.main()


def consulta(sql: str, **parametros) -> pd.DataFrame:
    """Consulta de solo lectura sobre la capa Gold."""
    engine = gold.motor()
    with engine.connect() as con:
        df = pd.read_sql(text(sql), con, params=parametros)
    engine.dispose()
    return df


def clp(valor) -> str:
    return f"${valor:,.0f}".replace(",", ".")


print("Python", sys.version.split()[0], "| pandas", pd.__version__, "| Gold:", Path(config.DWH_URL).name)

## 1. Fuentes y perfilado inicial

Antes de modelar, **perfilamos** las fuentes del lote 1 tal como llegan,
sin modificarlas, para medir los problemas de calidad que diagnosticamos en la EP1 de forma cualitativa.

In [ ]:
pd.DataFrame([{"archivo": p.name, "formato": p.suffix, "tamaño_KB": round(p.stat().st_size / 1024, 1)}
              for p in sorted((config.FUENTES / "lote_1").iterdir())])

In [ ]:
pos = pd.read_csv(config.FUENTES / "lote_1" / "pos_ventas_tiendas.csv", sep=";", dtype=str, keep_default_na=False)
display(pd.DataFrame({"vacíos": (pos == "").sum(), "valores distintos": pos.nunique()}).T)

patrones = {"dd/mm/aaaa hh:mm": r"^\d{2}/\d{2}/\d{4} \d{2}:\d{2}$",
            "aaaa-mm-dd hh:mm:ss": r"^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$",
            "dd-mm-aaaa hh:mm": r"^\d{2}-\d{2}-\d{4} \d{2}:\d{2}$"}
formatos = {nombre: int(pos["fecha_hora"].str.match(p).sum()) for nombre, p in patrones.items()}
formatos["no calza con ningún formato"] = len(pos) - sum(formatos.values())
print(f"Líneas POS: {len(pos):,}")
print("Formatos de fecha (validez):", formatos)
print("Precios con formato moneda '$19.990':", int(pos["precio_unitario"].str.startswith("$").sum()))
print("Precios negativos:", int(pos["precio_unitario"].str.startswith("-").sum()),
      "| precio 99.999.999:", int((pos["precio_unitario"] == "99999999").sum()))
print("Estados escritos de distintas formas (consistencia):", pos["estado"].value_counts().to_dict())
print("Ventas sin sucursal (completitud):", int((pos["cod_sucursal"] == "").sum()))
print("Líneas duplicadas exactas (unicidad):", int(pos.duplicated().sum()))

In [ ]:
with sqlite3.connect(config.FUENTES / "lote_1" / "crm_clientes.db") as conexion:
    crm = pd.read_sql("SELECT * FROM clientes", conexion)
rut = crm["rut"].map(rc.normalizar_rut)
print(f"Registros en el CRM: {len(crm):,}")
print("Formatos de RUT:", {"con puntos": int(crm["rut"].str.contains(r"\.").sum()),
                           "sin puntos con guion": int(crm["rut"].str.match(r"^\d+-[\dkK]$").sum()),
                           "sin guion": int(crm["rut"].str.match(r"^\d+[\dkK]$").sum())})
print("RUT repetidos al normalizar (clientes duplicados):", int(rut.dropna().duplicated().sum()))
print("RUT con dígito verificador inválido:", int((rut.notna() & ~rut.map(rc.rut_valido)).sum()))
print("Registros sin RUT:", int(rut.isna().sum()))
print("Correos mal formados:", int((~rc.email_valido(rc.normalizar_email(crm["email"]))).sum()))
print("Comuna vacía:", int((crm["comuna"] == "").sum()))

In [ ]:
pedidos = json.loads((config.FUENTES / "lote_1" / "ecommerce_pedidos.json").read_text(encoding="utf-8"))
items = [item for p in pedidos for item in p["items"]]
skus = set(pd.read_csv(config.FUENTES / "lote_1" / "equivalencias_codigos.csv")["sku_web"])
print(f"Pedidos: {len(pedidos):,} | líneas (items): {len(items):,} | "
      f"líneas con SKU sin equivalencia: {sum(i['sku_web'] not in skus for i in items)} | "
      f"pedidos sin fecha: {sum(p['fecha'] is None for p in pedidos)}")
print("Estructura anidada de un pedido:")
print(json.dumps({**pedidos[3], "cliente": {"email": "(oculto)", "rut": "(oculto)"}}, ensure_ascii=False, indent=1))

## 2. Carga inicial — lote 1 (año 2025)

Se reconstruyen las tres capas desde cero. El log (módulo `logging`, no `print`) muestra cada etapa:
conteos de Bronze, reglas aplicadas en Silver, carga de dimensiones y hecho en Gold y las 9 validaciones de cierre.
Gold y las validaciones corren en **una sola transacción**: si una validación falla, la carga completa se revierte.

In [ ]:
for capa in (config.BRONZE, config.SILVER):
    shutil.rmtree(capa, ignore_errors=True)   # partimos desde cero para que la evidencia sea reproducible
r1 = pipeline.ejecutar(1, reiniciar=True)
pd.Series(r1, name="lote 1").to_frame()

## 3. Capa Bronze — el dato tal como llegó, con linaje

Todo se guarda como texto y sin corregir, de modo que siempre se puede volver al registro original.
Cada fila suma metadatos de linaje: `_id_carga`, `_archivo`, `_fila_origen`, `_hash_archivo` y `_fecha_ingesta`.

In [ ]:
bronze_pos = pd.read_parquet(config.BRONZE / "pos_ventas" / "lote=1" / "datos.parquet")
print("Tipos de dato en Bronze:", bronze_pos.dtypes.astype(str).value_counts().to_dict())
display(bronze_pos.drop(columns=["rut_cliente"]).sample(3, random_state=7))

resumen = []
for carpeta in sorted(config.BRONZE.iterdir()):
    df = pd.read_parquet(carpeta / "lote=1" / "datos.parquet")
    resumen.append({"fuente": carpeta.name, "filas": len(df), "archivo": df["_archivo"].iat[0],
                    "sha256 (12)": df["_hash_archivo"].iat[0]})
pd.DataFrame(resumen)

## 4. Capa Silver — limpieza, homologación y tabla de rechazos

Se aplican las 22 reglas de calidad. Cada fila que no cumple va a `etl_rechazos` con su código y motivo:

* **RECHAZO**: el dato está mal y no puede usarse (fecha imposible, venta sin sucursal, precio negativo…).
* **EXCLUSION**: el dato es válido pero, por regla de negocio, no debe sumarse (línea duplicada, transacción anulada).

Ninguna fila se pierde sin dejar rastro: **leídas = válidas + rechazadas + excluidas** (validación V01).

In [ ]:
rechazos = consulta("""
    SELECT tipo, entidad, regla, motivo, COUNT(*) AS filas FROM etl_rechazos
    WHERE id_carga = :id GROUP BY tipo, entidad, regla, motivo ORDER BY tipo, regla""", id=r1["id_carga"])
display(rechazos)

datos = rechazos.sort_values("filas")
fig, ax = plt.subplots(figsize=(11, 4.8))
ax.barh(datos["regla"] + "  " + datos["motivo"], datos["filas"],
        color=["#9a6b3f" if t == "EXCLUSION" else "#b4531c" for t in datos["tipo"]])
ax.set_title("Lote 1 · filas rechazadas (rojo) y excluidas (café) por regla")
ax.set_xlabel("filas")
plt.tight_layout()
plt.show()

In [ ]:
ejemplo = consulta("""SELECT regla, motivo, archivo_origen, fila_origen, registro FROM etl_rechazos
                       WHERE id_carga = :id AND regla = 'VEN-05' LIMIT 1""", id=r1["id_carga"]).iloc[0]
print({k: ejemplo[k] for k in ("regla", "motivo", "archivo_origen", "fila_origen")})
print("Registro original guardado en la tabla de rechazos (datos personales enmascarados):")
print(json.dumps(json.loads(ejemplo["registro"]), ensure_ascii=False, indent=1))

In [ ]:
silver_ventas = pd.read_parquet(config.SILVER / "ventas")
casos = {
    "precio con formato moneda": bronze_pos["precio_unitario"].str.startswith("$"),
    "fecha aaaa-mm-dd": bronze_pos["fecha_hora"].str.match(r"^\d{4}-"),
    "fecha dd-mm-aaaa": bronze_pos["fecha_hora"].str.match(r"^\d{2}-"),
    "estado en minúsculas": bronze_pos["estado"].str.islower(),
    "estado con espacio final": bronze_pos["estado"].str.endswith(" "),
    "devolución": bronze_pos["estado"].str.strip().str.upper().eq("DEVUELTA"),
}
limpias = silver_ventas.assign(linea=silver_ventas["nro_linea"].astype(str))
filas = []
for problema, mascara in casos.items():
    cruce = bronze_pos[mascara].merge(limpias, left_on=["nro_boleta", "linea"], right_on=["numero_documento", "linea"],
                                      suffixes=("_bronze", "_silver")).head(1)
    filas.append(cruce.assign(problema=problema))
antes_despues = pd.concat(filas, ignore_index=True)
print("Antes (Bronze, texto crudo) y después (Silver, tipado y homologado):")
antes_despues[["problema", "nro_boleta", "fecha_hora_bronze", "fecha_hora_silver", "precio_unitario_bronze",
               "precio_unitario_silver", "estado", "es_devolucion"]].assign(estado=lambda d: "'" + d["estado"] + "'")

In [ ]:
base = config.SILVER / "ventas"
particiones = sorted(p.parent.relative_to(base).as_posix() for p in base.glob("anio=*/mes=*/*.parquet"))
print(f"Silver ventas particionado en {len(particiones)} carpetas:", particiones[:3], "...")

tamano_parquet = sum(p.stat().st_size for p in base.rglob("*.parquet"))
temporal = config.DATA / "_comparacion.csv"
silver_ventas.to_csv(temporal, index=False)
tamano_csv = temporal.stat().st_size
temporal.unlink()
print(f"Mismas {len(silver_ventas):,} filas -> CSV {tamano_csv / 1024:,.0f} KB vs Parquet {tamano_parquet / 1024:,.0f} KB "
      f"({tamano_csv / tamano_parquet:.1f}× más liviano)")

marzo = pd.read_parquet(base, filters=[("anio", "=", 2025), ("mes", "=", 3)])
print(f"Consulta de marzo 2025: lee solo 1 partición ({len(marzo):,} filas) en vez de las {len(silver_ventas):,}")

## 5. Capa Gold — el esquema estrella

Modelo derivado de la Bus Matrix de la EP1: proceso **Ventas**, grano **una línea de producto de cada boleta o pedido**,
cinco dimensiones conformadas, con claves sustitutas (en fecha, AAAAMMDD). El diagrama se genera desde el esquema real de la base
(`python -m src.documentacion`) y también se exporta a `docs/modelo_estrella.drawio` para editarlo en Draw.io.

![Modelo estrella](../docs/modelo_estrella.png)

In [ ]:
tablas = ["dim_fecha", "dim_canal", "dim_sucursal", "dim_producto", "dim_cliente", "fact_ventas",
          "etl_rechazos", "etl_validaciones", "etl_control_cargas", "agg_ventas_mes_region"]
pd.DataFrame({"tabla": tablas, "filas": [consulta(f"SELECT COUNT(*) AS n FROM {t}")["n"].iat[0] for t in tablas]})

In [ ]:
display(consulta("""SELECT fecha, canal, sucursal, region_sucursal, producto, categoria, segmento_cliente,
                            unidades, monto_neto, venta_neta FROM vw_ventas LIMIT 5"""))

por_region = consulta("""SELECT region_sucursal AS region, canal, SUM(venta_neta) AS venta_neta
                           FROM vw_ventas WHERE anio = 2025 GROUP BY region_sucursal, canal""")
tabla = por_region.pivot(index="region", columns="canal", values="venta_neta").fillna(0)
tabla = tabla.loc[tabla.sum(axis=1).sort_values().index] / 1e6
ax = tabla.plot.barh(stacked=True, figsize=(9, 4.8), color=["#b8860b", "#1f3a5f", "#2a7f62"])
ax.set_title("Venta neta 2025 por región de la sucursal y canal (millones de $)")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

## 6. Carga incremental — lote 2 (enero a junio 2026) y Slowly Changing Dimensions

El lote 2 trae ventas nuevas y también cambios en los maestros: clientes que se mudan o cambian de segmento,
productos reclasificados, precios y marcas corregidos, una sucursal nueva y una dirección corregida.
Cada tipo de cambio se gestiona con la técnica SCD que corresponde a lo que el negocio necesita recordar.

In [ ]:
r2 = pipeline.ejecutar(2)
pd.DataFrame({"lote 1": r1, "lote 2": r2}).drop(index="id_carga")

### SCD tipo 2 — Cliente (comuna/región y segmento)

Cuando un cliente se muda o cambia de segmento se **cierra** su versión vigente y se **abre** una nueva con su propia
clave sustituta y su vigencia. Las ventas se enlazan a la versión vigente **en la fecha de la venta**,
así las ventas de 2025 conservan la región que el cliente tenía en ese momento.

In [ ]:
caso = consulta("""
    SELECT cl.id_cliente FROM dim_cliente cl JOIN fact_ventas f ON f.sk_cliente = cl.sk_cliente
    WHERE cl.id_cliente IN (SELECT id_cliente FROM dim_cliente WHERE version = 2)
    GROUP BY cl.id_cliente
    HAVING COUNT(DISTINCT cl.sk_cliente) = 2 AND COUNT(DISTINCT cl.region) = 2
    ORDER BY cl.id_cliente LIMIT 1""")["id_cliente"].iat[0]
print("Versiones del cliente", caso, "(id seudonimizado):")
display(consulta("""SELECT sk_cliente, version, comuna, region, segmento, fecha_inicio_vigencia,
                            fecha_fin_vigencia, es_actual FROM dim_cliente WHERE id_cliente = :id ORDER BY version""", id=caso))
print("Sus compras y la versión a la que quedó enlazada cada una:")
consulta("""SELECT d.fecha, f.numero_documento, f.sk_cliente, cl.version, cl.region AS region_al_comprar, f.monto_neto
            FROM fact_ventas f JOIN dim_cliente cl ON cl.sk_cliente = f.sk_cliente
            JOIN dim_fecha d ON d.sk_fecha = f.sk_fecha WHERE cl.id_cliente = :id ORDER BY d.fecha""", id=caso)

In [ ]:
display(consulta("""SELECT version, COUNT(*) AS clientes FROM dim_cliente WHERE sk_cliente <> -1
                       GROUP BY version""").rename(columns={"version": "versiones del cliente"}))
impacto = consulta("""
    SELECT COUNT(*) AS ventas, SUM(f.monto_neto) AS monto
    FROM fact_ventas f
    JOIN dim_cliente cl     ON cl.sk_cliente = f.sk_cliente
    JOIN dim_cliente actual ON actual.id_cliente = cl.id_cliente AND actual.es_actual = 1
    WHERE cl.es_actual = 0 AND cl.region <> actual.region""").iloc[0]
print(f"{int(impacto.ventas):,} ventas por {clp(impacto.monto)} conservan la región del cliente al momento de comprar. "
      "Con SCD tipo 1 (sobrescribir) se habrían reasignado a su región actual y el análisis histórico quedaría mal.")

### SCD tipo 3 — Categoría del producto

En 2026 la línea **Smartwatches** pasa de *Accesorios* a *Tecnología*. Con SCD tipo 3 se guarda la categoría
vigente y la anterior en columnas separadas: así se puede reportar con cualquiera de las dos clasificaciones
y comparar *antes y después* de la reclasificación.

In [ ]:
display(consulta("""SELECT cod_producto, nombre_producto, subcategoria, categoria_anterior, categoria,
                            fecha_cambio_categoria FROM dim_producto WHERE categoria_anterior IS NOT NULL LIMIT 5"""))
consulta("""SELECT anio,
                   SUM(CASE WHEN categoria = 'Tecnología' THEN venta_neta END) AS tecnologia_clasificacion_vigente,
                   SUM(CASE WHEN COALESCE(categoria_anterior, categoria) = 'Tecnología' THEN venta_neta END)
                       AS tecnologia_clasificacion_anterior
            FROM vw_ventas GROUP BY anio""")

### SCD tipo 1 — Correcciones sin historia

Los errores corregidos (marca mal escrita, dirección equivocada) y los atributos cuya historia no interesa
(precio de lista vigente) se **sobrescriben**. Gold no guarda el valor anterior, pero **Bronze sí**:
la trazabilidad no se pierde.

In [ ]:
display(consulta("""SELECT cod_sucursal, nombre_sucursal, direccion, fecha_actualizacion, id_carga
                       FROM dim_sucursal WHERE cod_sucursal IN ('SUC-04', 'SUC-13')"""))

lote1 = pd.read_parquet(config.BRONZE / "maestro_productos" / "lote=1" / "datos.parquet")[["cod_producto", "marca", "precio_lista"]]
actual = consulta("SELECT cod_producto, nombre_producto, marca, precio_lista FROM dim_producto")
comparacion = lote1.merge(actual, on="cod_producto", suffixes=("_bronze_lote1", "_gold"))
comparacion["precio_lista_bronze_lote1"] = comparacion["precio_lista_bronze_lote1"].astype(int)
cambiados = comparacion[(comparacion["marca_bronze_lote1"] != comparacion["marca_gold"])
                        | (comparacion["precio_lista_bronze_lote1"] != comparacion["precio_lista_gold"])]
print(f"{len(cambiados)} productos con marca o precio corregidos (SCD tipo 1). Ejemplos:")
cambiados.head(8)

## 7. Idempotencia — volver a ejecutar no duplica

Se ejecuta otra vez el lote 2. La clave natural `(fuente_origen, numero_documento, nro_linea)` impide insertar
dos veces la misma línea y las dimensiones solo cambian si llega un cambio real.

In [ ]:
antes = consulta("SELECT COUNT(*) AS filas, SUM(monto_neto) AS monto_neto FROM fact_ventas").iloc[0]
r3 = pipeline.ejecutar(2)
despues = consulta("SELECT COUNT(*) AS filas, SUM(monto_neto) AS monto_neto FROM fact_ventas").iloc[0]
print(f"Re-ejecución: {r3['fact_insertadas']} líneas insertadas, {r3['fact_omitidas']:,} ya existían; "
      f"nuevas versiones SCD2: {r3['clientes_scd2_nuevas_versiones']}")
pd.DataFrame({"antes": antes, "después": despues})

## 8. Manejo de errores — una fuente que rompe el contrato

Se simula un lote 3 cuyo archivo POS llega **sin la columna `estado`**. El contrato de datos lo detecta en Bronze,
el pipeline **falla de forma visible** (log ERROR), la carga queda registrada como `FALLIDA` en `etl_control_cargas`
y Gold no se modifica.

In [ ]:
lote_roto = config.FUENTES / "lote_3"
shutil.copytree(config.FUENTES / "lote_2", lote_roto, dirs_exist_ok=True)
pos_roto = pd.read_csv(lote_roto / "pos_ventas_tiendas.csv", sep=";", dtype=str).drop(columns=["estado"])
pos_roto.to_csv(lote_roto / "pos_ventas_tiendas.csv", sep=";", index=False)

filas_antes = consulta("SELECT COUNT(*) AS n FROM fact_ventas")["n"].iat[0]
try:
    pipeline.ejecutar(3)
except bronze.ContratoError as exc:
    print("Error capturado por el orquestador:", exc)
finally:
    shutil.rmtree(lote_roto)
print("Filas en fact_ventas antes / después del intento:", filas_antes, "/",
      consulta("SELECT COUNT(*) AS n FROM fact_ventas")["n"].iat[0])
consulta("""SELECT id_carga, lote, estado, lineas_leidas, lineas_rechazadas, lineas_excluidas,
                   filas_insertadas_fact, filas_omitidas_ya_cargadas, version_codigo, mensaje
            FROM etl_control_cargas ORDER BY inicio""").convert_dtypes()

## 9. Validaciones de cierre y trazabilidad extremo a extremo

Las 9 validaciones corren antes de confirmar cada carga y quedan registradas en `etl_validaciones`.
Después se sigue **una cifra del hecho hasta la línea exacta del archivo fuente** (Gold → Bronze → archivo).

In [ ]:
consulta("SELECT validacion, resultado, detalle FROM etl_validaciones WHERE id_carga = :id", id=r2["id_carga"])

In [ ]:
fila = consulta("""
    SELECT f.sk_venta, d.fecha, p.nombre_producto, s.nombre_sucursal, f.cantidad, f.precio_unitario, f.descuento,
           f.monto_neto, f.id_carga, f.archivo_origen, f.fila_origen, c.lote
    FROM fact_ventas f
    JOIN dim_fecha d    ON d.sk_fecha = f.sk_fecha
    JOIN dim_producto p ON p.sk_producto = f.sk_producto
    JOIN dim_sucursal s ON s.sk_sucursal = f.sk_sucursal
    JOIN etl_control_cargas c ON c.id_carga = f.id_carga
    WHERE f.fuente_origen = 'POS' AND f.descuento > 0 ORDER BY f.sk_venta LIMIT 1""").iloc[0]
print("1) Gold · fact_ventas")
display(fila.to_frame().T)

registro = pd.read_parquet(config.BRONZE / "pos_ventas" / f"lote={fila.lote}" / "datos.parquet")
registro = registro[registro["_fila_origen"] == str(fila.fila_origen)].assign(
    rut_cliente=lambda d: d["rut_cliente"].map(rc.enmascarar))
print("2) Bronze · registro crudo con sus metadatos de linaje")
display(registro)

linea = linecache.getline(str(config.FUENTES / f"lote_{fila.lote}" / fila.archivo_origen), int(fila.fila_origen)).strip()
campos = linea.split(";")
campos[4] = rc.enmascarar(campos[4])
print(f"3) Archivo fuente {fila.archivo_origen}, línea {fila.fila_origen}:")
print("   ", ";".join(campos))

## 10. Optimización y capa semántica para IA

* **Índices** en las claves foráneas del hecho: el motor busca por índice en vez de recorrer la tabla.
* **Agregado precalculado** `agg_ventas_mes_region` (emula una vista materializada; se refresca en cada carga).
* **Capa semántica** (`docs/capa_semantica.json`): nombres, sinónimos, métricas oficiales y reglas para que un
  asistente NLQ/text-to-SQL genere SQL correcto, siempre con **validación humana**.

In [ ]:
for sql in ["SELECT SUM(monto_neto) FROM fact_ventas WHERE sk_producto = 25",
            "SELECT SUM(monto_neto) FROM fact_ventas WHERE sk_fecha BETWEEN 20260301 AND 20260331"]:
    print(sql, "\n   plan ->", consulta("EXPLAIN QUERY PLAN " + sql)["detail"].tolist())


def medir(sql: str, repeticiones: int = 10) -> float:
    engine = gold.motor()
    with engine.connect() as con:
        inicio = time.perf_counter()
        for _ in range(repeticiones):
            con.execute(text(sql)).fetchall()
        duracion = (time.perf_counter() - inicio) / repeticiones * 1000
    engine.dispose()
    return duracion


t_vista = medir("SELECT anio_mes, region_sucursal, canal, SUM(venta_neta) FROM vw_ventas GROUP BY 1, 2, 3")
t_agregado = medir("SELECT anio_mes, region, canal, venta_neta FROM agg_ventas_mes_region")
print(f"\nKPI venta neta por mes, región y canal: vista detalle {t_vista:.1f} ms vs agregado precalculado "
      f"{t_agregado:.2f} ms (≈{t_vista / t_agregado:.0f}× más rápido)")

In [ ]:
semantica = json.loads((config.DOCS / "capa_semantica.json").read_text(encoding="utf-8"))
print("Métricas oficiales:", ", ".join(semantica["metricas"]))
for ejemplo in semantica["ejemplos_nlq"]:
    print("\nPregunta:", ejemplo["pregunta"])
    print("SQL generado desde la capa semántica:", ejemplo["sql"])
    display(consulta(ejemplo["sql"]))

resultado = consulta(semantica["ejemplos_nlq"][0]["sql"]).iat[0, 0]
oficial = consulta("SELECT venta_neta FROM vw_kpi_mensual WHERE canal = 'App móvil' AND anio_mes = '2026-03'").iat[0, 0]
print(f"\nChecklist de validación humana · contrastar con una cifra conocida: respuesta {clp(resultado)} "
      f"vs KPI oficial {clp(oficial)} -> {'coincide, se puede comunicar' if resultado == oficial else 'NO coincide: no comunicar'}")

## 11. Primeras respuestas a las preguntas de negocio

Con Gold cargado ya podemos responder las preguntas descriptivas que planteamos en la EP1. Los dashboards de la etapa de visualización
se construirán sobre estas mismas vistas.

**¿Qué sucursales, canales o categorías explican mejor o peor desempeño de ventas?**

In [ ]:
kpi = consulta("SELECT * FROM vw_kpi_mensual")
mensual = kpi.pivot(index="anio_mes", columns="canal", values="venta_neta") / 1e6
ax = mensual.plot(figsize=(10, 4), marker="o", ms=3, color=["#b8860b", "#1f3a5f", "#2a7f62"])
ax.set_title("Venta neta mensual por canal (millones de $)")
ax.set_xlabel("")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

display(consulta("""SELECT canal, anio, SUM(venta_neta) AS venta_neta,
                           ROUND(SUM(venta_neta) * 1.0 / COUNT(DISTINCT CASE WHEN es_devolucion = 0 THEN numero_documento END)) AS ticket_promedio
                    FROM vw_ventas GROUP BY canal, anio ORDER BY canal, anio"""))
display(consulta("""SELECT sucursal, region_sucursal, SUM(venta_neta) AS venta_neta_2025 FROM vw_ventas
                    WHERE anio = 2025 AND canal = 'Tienda física' GROUP BY sucursal, region_sucursal
                    ORDER BY venta_neta_2025 DESC""").iloc[[0, 1, -2, -1]])
consulta("""SELECT categoria, subcategoria, SUM(venta_neta) AS venta_neta,
                   ROUND(SUM(monto_devuelto) * 100.0 / NULLIF(SUM(venta_neta) + SUM(monto_devuelto), 0), 1) AS pct_devolucion
            FROM vw_ventas GROUP BY categoria, subcategoria ORDER BY venta_neta DESC LIMIT 8""")

**¿Cómo se comportan los clientes según recurrencia, canal o segmento?**

In [ ]:
recurrencia = consulta("""
    SELECT CASE WHEN compras = 1 THEN '1 compra' WHEN compras <= 3 THEN '2 a 3 compras'
                WHEN compras <= 6 THEN '4 a 6 compras' ELSE '7 o más' END AS recurrencia_2025,
           COUNT(*) AS clientes, ROUND(AVG(venta)) AS venta_promedio_por_cliente
    FROM (SELECT id_cliente, COUNT(DISTINCT numero_documento) AS compras, SUM(venta_neta) AS venta
          FROM vw_ventas WHERE anio = 2025 AND es_cliente_anonimo = 0 AND es_devolucion = 0 GROUP BY id_cliente)
    GROUP BY 1 ORDER BY MIN(compras)""")
display(recurrencia)
consulta("""SELECT segmento_cliente, canal, COUNT(DISTINCT id_cliente) AS clientes, SUM(venta_neta) AS venta_neta
            FROM vw_ventas WHERE es_cliente_anonimo = 0 AND anio = 2025
            GROUP BY segmento_cliente, canal ORDER BY segmento_cliente, venta_neta DESC""")

**¿Qué indicadores deben seguir las áreas, y cómo aseguramos que sean confiables y consistentes?**
Las métricas oficiales viven en `docs/capa_semantica.json` y en las vistas: una sola definición de venta neta,
monto devuelto, tasa de devolución, unidades vendidas, documentos, ticket promedio y clientes activos. Su confiabilidad
la sostienen las 9 validaciones de cierre, la tabla de rechazos y el linaje de cada fila. Para la IA generativa, la capa
semántica fija qué vistas consultar y cómo, y toda respuesta pasa por el checklist de validación humana (sección 10).

## 12. Control de versiones

Cada carga registra el commit de Git con el que se ejecutó (`version_codigo`), lo que une la trazabilidad
del dato con la trazabilidad del código.

In [ ]:
historial = subprocess.run(["git", "log", "--oneline", "-n", "15"], cwd=RAIZ, capture_output=True, text=True)
print(historial.stdout or "El repositorio aún no tiene commits.")
consulta("SELECT id_carga, estado, version_codigo FROM etl_control_cargas ORDER BY inicio")

## Conclusión

El pipeline transforma siete fuentes heterogéneas y con fallas de calidad en un esquema estrella confiable:
cada línea leída queda contabilizada (válida, rechazada o excluida), las dimensiones gestionan su evolución con
SCD 1, 2 y 3, la carga es idempotente y transaccional, y cada venta de Gold puede rastrearse hasta la línea
exacta de su archivo fuente. Gold queda listo para el modelo semántico y los dashboards de la etapa de visualización.